In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

/Users/yohanshah/Documents/what/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import sys
print(sys.executable)

/Users/yohanshah/Documents/what/.venv/bin/python


In [3]:
# 1. MODEL SELECTION & INFRASTRUCTURE SETUP

# Specify the model identifier from Hugging Face Hub.
MODEL_NAME = "google/flan-t5-base"

print(f"[INFO] Initializing environment and fetching open-source weights for: {MODEL_NAME}...")

[INFO] Initializing environment and fetching open-source weights for: google/flan-t5-base...


In [4]:
# 2 Tokenizer $ Model Loading 

# A TTokenizer :
# Neural networks wwork wih numerical matrices, not raw string characters.
#the tokenizer splits human text intpo sub-word tokeens and maps them to uniqque integer IDs.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [5]:
#B. Model :
#Load the neiural network architecture along with its pre-trained w\eight matrices into memory.
#torch_dtype= torch.float32 ensure standard 32-bit floarting point precision, providing 
#full numerical precision and broad compability across CPUs and standard GPUs.
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32
)

print(f"[INFO] Model sucessfully instialized in memory with vocabulary size:{tokenizer.vocab_size}tokens.")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 282/282 [00:00<00:00, 6598.31it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


[INFO] Model sucessfully instialized in memory with vocabulary size:32100tokens.


In [26]:
def generate_text_response(prompt_text: str) -> str:
    """
    Generate text using model.generate().
    """

    # Put model in evaluation mode
    model.eval()

    # Tokenize input
    inputs = tokenizer(prompt_text, return_tensors="pt")

    # Disable gradient calculation during inference
    with torch.no_grad():

        output_ids = model.generate(
            **inputs,

            # Length controls
            max_new_tokens=256,
            min_new_tokens=50,

            # Beam search
            num_beams=4,
            early_stopping=True,

            # Decoding
            do_sample=False,

            # Sampling parameters (only used if do_sample=True)
            temperature=0.8,
            top_k=50,
            top_p=0.9,

            # Repetition control
            repetition_penalty=2.0,
            no_repeat_ngram_size=4
        )

    # Decode generated tokens
    generated_text = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

    return generated_text

In [27]:
# Step 3: Detokenize and cleanup
# Step 1: Tokenize the input
inputs = tokenizer(
   "Answer the following question: What is artificial intelligence?",
    return_tensors="pt"
)

# Step 2: Generate output token IDs
output_ids = model.generate(
    **inputs,
    max_new_tokens=100)

# Step 3: Decode token IDs
#skip_text = True' Strip out control tokens(eg.., <pad> , <s>, <unk>).
generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

generated_text



'Artificial intelligence is a technology that uses artificial intelligence to learn and learn from human beings.'

In [28]:
# Benchmark Case 1: Document Summarization

document_content = (
    "Generative AI, particularly Large Language Models (LLMs), represents a fundamental paradigm shift in machine learning. "
    "Unlike discriminative architectures that evaluate features to assign decision boundaries, generative models learn "
    "the joint probability distribution across massive datasets such as text, images, and audio to synthesize entirely "
    "new, semantically coherent content. This capability enables tasks such as creative writing, code generation, "
    "image synthesis, and sophisticated summarization. Their architectural advantage lies in understanding and "
    "replicating complex data patterns, making them highly versatile across industries, from content creation to "
    "scientific research."
)

document_prompt = f"summarize: {document_content}"

print("INPUT PROMPT (Summarization Task):")
print(document_prompt)

summary_result = generate_text_response(document_prompt)

print("\nMODEL RESPONSE:")
print(summary_result)

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


INPUT PROMPT (Summarization Task):
summarize: Generative AI, particularly Large Language Models (LLMs), represents a fundamental paradigm shift in machine learning. Unlike discriminative architectures that evaluate features to assign decision boundaries, generative models learn the joint probability distribution across massive datasets such as text, images, and audio to synthesize entirely new, semantically coherent content. This capability enables tasks such as creative writing, code generation, image synthesis, and sophisticated summarization. Their architectural advantage lies in understanding and replicating complex data patterns, making them highly versatile across industries, from content creation to scientific research.

MODEL RESPONSE:
Generative AI, particularly Large Language Models (LLMs), represents a fundamental paradigm shift in machine learning. Unlike discriminative architectures that evaluate features to assign decision boundaries, generative models learn the joint pro

In [31]:
# -- Benchmark case 2 : conceptual q and a // Reasoning --

reasoning_quesion = (
    "what is the main architectural advantages of hosting open-sourcew LLMs"
    " on local infrastructure compared to using closed-source APIs ?"
)



# Explicitly prepending 'question:' guides the model to invoke its instruction-tuned q&a weights.
reasoning_prompt = f"question: {reasoning_quesion}"


print(f"INPUT PROMPT (Technical Q & A Tasks):\n{reasoning_prompt}")
reasoning_result = generate_text_response(reasoning_prompt)
print(f"\nMODEL RESPONSE 2:\n{reasoning_result}")

INPUT PROMPT (Technical Q & A Tasks):
question: what is the main architectural advantages of hosting open-sourcew LLMs on local infrastructure compared to using closed-source APIs ?

MODEL RESPONSE 2:
open-sourcew LLMs on local infrastructure compared to closed-source APIs are more efficient and reliable than the closed-source architectures that have been used in the past. Open-sourceW LLMs do not require any hardware or software to be installed on the local infrastructure
